#TRILLsson FEATURE EXTRACTION PIPELINE
### ASR - Project Radboud University
It takes preprocessed patient-only Wav files and saves TRILLsson embeddings

In [ ]:
# !pip install -U tensorflow tensorflow-hub librosa soundfile tqdm pandas

In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import librosa
from tqdm import tqdm

In [ ]:
# mounting Google Drive
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)
# main project directory
PROJECT = Path("/content/drive/MyDrive/asr project")



# for MCI / patients only folder definition
PREPROCESSED_ROOT = PROJECT / "preprocessed_patient_audio" / "Patients"
FEATURE_ROOT = PROJECT / "features" / "trillsson_patient_only"

assert PROJECT.exists(), f"Project folder not found: {PROJECT}"
assert PREPROCESSED_ROOT.exists(), f"Preprocessed folder not found: {PREPROCESSED_ROOT}"

FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT)
print("Preprocessed audio root:", PREPROCESSED_ROOT)
print("Feature output root:", FEATURE_ROOT)




# load TRILLsson model

TRILLSSON_URL = "https://tfhub.dev/google/trillsson1/1"
print("Loading TRILLsson model...")
trillsson_model = hub.load(TRILLSSON_URL)
print("Loaded TRILLsson model.")



# audio loading function
def load_wav_mono_16k_librosa(wav_path, target_sr=16000):
    """
    Load audio file and prepare it for TRILLsson embedding extraction.
    Processing steps:
    1. Load the audio file with Librosa.
    2. Convert stereo or multi-channel audio to mono.
    3. Resample the audio to the target sample rate, normally 16 kHz.
    4. Convert the waveform to NumPy float32 format.
    5. Peak-normalize the waveform to the range [-1, 1].

    Parameters:
     wav_path: path to the input WAV file.
     target_sr: Target audio sampling rate in hertz.

    Returns:
     A NumPy array containing the prepared audio waveform.

    """
    # load audio file
    x, sr = librosa.load(
        str(wav_path),
        sr=target_sr,
        mono=True
    )
    # convert the waveform to float32
    x = x.astype(np.float32)

    # find the largest absolute amplitude in the waveform
    max_abs = np.max(np.abs(x)) if len(x) > 0 else 0.0


    # peak-normalize non-silent audio.
    # silent audio is left unchanged to avoid division by zero
    if max_abs > 0:
        x = x / max_abs

    return x


# segmenting function
def split_into_segments(waveform, sample_rate=16000, segment_seconds=10):
    """
    Audio is split into audio segments of a fixed duration.
    Each has a duration of `segment_seconds`. If the final
    segment is shorter than the required length, it is padded with
    zeros.

    Parameters:
    waveform : np.ndarray, 1D audio waveform
        One-dimensional audio waveform.

    sample_rate : int, sampling rate in hertz
    segment_seconds : float, duration of each segment in seconds


    Returns: a list/np.ndarray of segments equal lenght
    """
    # calculate required number of samples per segment
    segment_len = int(sample_rate * segment_seconds)

    # empty waveform is replaced by one full- lenght silent segment
    if len(waveform) == 0:
        waveform = np.zeros(segment_len, dtype=np.float32)

    # store audio segments
    segments = []

    # iterate through waveform one segment at the time
    for start in range(0, len(waveform), segment_len):
        # sample from current segment
        segment = waveform[start:start + segment_len]

        # pad the final segment with 0s if it is short
        if len(segment) < segment_len:
            segment = np.pad(
                segment,
                (0, segment_len - len(segment)),
                mode="constant"
            )

        # store it into array
        segments.append(segment.astype(np.float32))

    # fallback for at least one segment returned
    if len(segments) == 0:
        segments = [np.zeros(segment_len, dtype=np.float32)]

    return segments


# TRILLsson extraction function

def extract_trillsson(
    wav_path,
    model,
    sample_rate=16000,
    segment_seconds=10
):
    """
    Extract TRILLsson embeddings from a WAV file.

    Parameters:
      wav_path: path to the input WAV file
      model: TRILLsson model loaded with tensorflow_hub.load
      sample_rate: target audio sampling rate in hertz
      segment_seconds: duration of each segment in seconds

    Returns:
        TRILLsson vector as numpy array, usually shape (1024,)
    """
    # load the audio as a mono, peak-normalised float32 waveform
    waveform = load_wav_mono_16k_librosa(
        wav_path,
        target_sr=sample_rate
    )

    # divide into fixed segments
    segments = split_into_segments(
        waveform,
        sample_rate=sample_rate,
        segment_seconds=segment_seconds
    )

    #store embeddings for each audio segment
    embeddings = []

    #Iteraate over the segments
    for segment in segments:
       # add batch dimention
        segment_tf = tf.convert_to_tensor(
            segment[None, :],
            dtype=tf.float32
        )
        # the audio segment through the TRILLsson model.
        output = model(segment_tf)


        # remove the batch dim
        emb = output["embedding"].numpy().squeeze(0)
        embeddings.append(emb)

    # stack all embeddings and average across time
    # one fixed-size vector for complete audio file
    trillsson_vec = np.mean(
        np.stack(embeddings),
        axis=0
    ).astype(np.float32)
    return trillsson_vec



# find all patient-only WAV files
patient_wavs = sorted(PREPROCESSED_ROOT.rglob("*_patient.wav"))
print("Found patient-only WAVs:", len(patient_wavs))

# print example
print("\nExample files:")
for p in patient_wavs[:10]:
    print(p)

assert len(patient_wavs) > 0, "No *_patient.wav files found. Check preprocessing output."



# extract and save TRILLsson features
feature_rows = []
#iterate over all patient waves
for wav_path in tqdm(patient_wavs):
    wav_path = Path(wav_path)

    # Expected structure:
    # preprocessed_patient_audio / Patients / DatasetName / wav_patient_only / file_patient.wav
    dataset_name = wav_path.parents[1].name
    file_id = wav_path.stem.replace("_patient", "")

    # set output directory
    out_dir = FEATURE_ROOT / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{file_id}_trillsson.npy"

    try:
        # reuse an existing feature file instead of extracting it again
        if out_path.exists():
            trillsson_vec = np.load(out_path)
            status = "already_exists"
        else:
            # extract one file-level TRILLsson embedding
            trillsson_vec = extract_trillsson(
                wav_path=wav_path,
                model=trillsson_model,
                sample_rate=16000,
                segment_seconds=10
            )
            # Save the embedding as a NumPy array.
            np.save(out_path, trillsson_vec)

            # safety check
            if not out_path.exists():
                raise FileNotFoundError(f"Saved file not found: {out_path}")

            # reload to verify it can be accessed
            trillsson_vec = np.load(out_path)

            status = "ok"

        # meta data for processed file
        feature_rows.append({
            "dataset": dataset_name,
            "file_id": file_id,
            "wav_path": str(wav_path),
            "trillsson_path": str(out_path),
            "trillsson_shape": str(tuple(trillsson_vec.shape)),
            "status": status,
            "error": error
        })

    except Exception as e:
        # record the error without stopping the entire extraction loop
        feature_rows.append({
            "dataset": dataset_name,
            "file_id": file_id,
            "wav_path": str(wav_path),
            "trillsson_path": str(out_path),
            "trillsson_shape": "",
            "status": "error",
            "error": str(e)
        })



# save feature metadata CSV
trillsson_metadata = pd.DataFrame(feature_rows)
metadata_path = FEATURE_ROOT / "trillsson_feature_metadata.csv"
trillsson_metadata.to_csv(metadata_path, index=False)

print("\nSaved TRILLsson metadata to:", metadata_path)
print("Total files:", len(trillsson_metadata))
print("\nStatus counts:")
print(trillsson_metadata["status"].value_counts())
print("\nBy dataset/status:")
print(trillsson_metadata.groupby(["dataset", "status"]).size())


# check one saved TRILLsson vector
# keep status for dedugging
valid_df = trillsson_metadata[
    trillsson_metadata["status"].isin(["ok", "already_exists"])
].copy()

# check if the feature exist in the stored path location
valid_df["feature_exists"] = valid_df["trillsson_path"].apply(lambda p: Path(p).exists())

# rows where features are missing
missing_features = valid_df[~valid_df["feature_exists"]]
print("\nMissing feature files among valid rows:", len(missing_features))
# display those problematic ones
if len(missing_features) > 0:
    display(missing_features[["dataset", "file_id", "trillsson_path", "wav_path"]].head(30))

# remove all not exisiting
valid_df = valid_df[valid_df["feature_exists"]].copy()
assert len(valid_df) > 0, "No valid TRILLsson features were extracted."

# select the first file of them
example_path = valid_df.iloc[0]["trillsson_path"]
example_vec = np.load(example_path)

print("\nExample TRILLsson path:", example_path)
print("Example TRILLsson shape:", example_vec.shape)
print("First 10 values:")
print(example_vec[:10])


# save combined feature CSV
rows = []

#load the valid files
for _, row in valid_df.iterrows():
    # load embeddings for current audio
    trillsson_vec = np.load(row["trillsson_path"])

    # Create the non-feature columns for the current CSV row.
    feature_row = {
        "dataset": row["dataset"],
        "file_id": row["file_id"],
        "wav_path": row["wav_path"],
        "trillsson_path": row["trillsson_path"],
    }

    # add every element from the vectors as a separate column
    for i, value in enumerate(trillsson_vec):
        feature_row[f"trillsson_{i}"] = float(value)

    rows.append(feature_row)

# convert all feature records into a Pandas DataFrame.
# every row is one audio file
# each embedding dim is a separare column
trillsson_df = pd.DataFrame(rows)

combined_csv_path = FEATURE_ROOT / "trillsson_features_combined.csv"
# store the data frame
trillsson_df.to_csv(combined_csv_path, index=False)
print("\nSaved combined TRILLsson CSV to:", combined_csv_path)
print("Combined CSV shape:", trillsson_df.shape)



# save the combined feature matrix as an NPZ archive
X = []
dataset = []
file_id = []
wav_paths = []
trillsson_paths = []

for _, row in valid_df.iterrows():
    # load embeddings
    trillsson_vec = np.load(row["trillsson_path"])

    # add embeddings to feature list
    X.append(trillsson_vec)
    # store meta data
    dataset.append(row["dataset"])
    file_id.append(row["file_id"])
    wav_paths.append(row["wav_path"])
    trillsson_paths.append(row["trillsson_path"])
# stack all one dim embeddings
# for N files with 1024-dimensional embeddings, the resulting # matrix will have shape: # # (N, 1024)
X = np.vstack(X).astype(np.float32)

npz_path = FEATURE_ROOT / "trillsson_features_matrix.npz"

# save the data structure
np.savez(
    npz_path,
    X=X,
    dataset=np.array(dataset),
    file_id=np.array(file_id),
    wav_path=np.array(wav_paths),
    trillsson_path=np.array(trillsson_paths)
)

# Final check of locations and shape
print("\nSaved combined TRILLsson matrix to:", npz_path)
print("X shape:", X.shape)

print("\nDone.")
print("Feature folder:", FEATURE_ROOT)
print("Metadata CSV:", metadata_path)
print("Combined CSV:", combined_csv_path)
print("Combined NPZ:", npz_path)